# Other organisms — deep dive (mouse + pig)

This notebook is a marketing appendix that strengthens the claim that IDTrack’s core semantics generalize beyond human.

## Rationale

The main manuscript focuses on human, but a reviewer may ask whether the approach is human-specific.
This notebook provides *bounded, cache-first* evidence that for other Ensembl organisms:

- The graph snapshot can be reused from the shared local repository
- Outcome semantics (1→0 / 1→1 / 1→n) behave as expected
- The time axis (target release sweep) produces stable, interpretable trends

Scope: within-species conversions only (no ortholog mapping).

## Outputs

- `idtrack-manuscript/figures/fig_other_organisms_deep_dive.pdf`
- Cached summaries under `idtrack/docs/_notebooks/idtrack_cache/experiments/other_organisms_deep_dive/`


In [ ]:
from __future__ import annotations

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

try:
    import seaborn as sns
except Exception:  # noqa: S110
    sns = None

import sys

# Add experiments/src to sys.path
REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT != REPO_ROOT.parent and not ((REPO_ROOT / 'idtrack').is_dir() and (REPO_ROOT / 'idtrack-manuscript').is_dir()):
    REPO_ROOT = REPO_ROOT.parent

EXPERIMENTS_SRC = REPO_ROOT / 'idtrack' / 'reproducibility' / 'experiments' / 'src'
sys.path.append(str(EXPERIMENTS_SRC))

from experiments_utils import (  # noqa: E402
    MANUSCRIPT_COLORS,
    experiments_cache_dir,
    idtrack_cache_dir,
    manuscript_figures_dir,
    read_pickle,
    write_pickle,
)

if sns is not None:
    sns.set_theme(style='whitegrid', context='paper')

plt.rcParams.update({'savefig.dpi': 300, 'figure.dpi': 140})

IDTRACK_LOCAL_REPO = idtrack_cache_dir(REPO_ROOT)
CACHE_DIR = experiments_cache_dir(REPO_ROOT, experiment='other_organisms_deep_dive')
MANUSCRIPT_FIGURES = manuscript_figures_dir(REPO_ROOT)

print('IDTRACK_LOCAL_REPO:', IDTRACK_LOCAL_REPO)
print('CACHE_DIR:', CACHE_DIR)


In [ ]:
# -------------------- Configuration --------------------

ORGANISMS = [
    {'alias': 'mouse', 'prefix': 'ENSMUSG'},
    {'alias': 'pig', 'prefix': 'ENSSSCG'},
]

SNAPSHOT_RELEASE = 114  # must be <= what you have cached
TARGET_RELEASES = list(range(100, 115, 3))  # sweep for the time-axis plot

N_SAMPLE_IDS = 300
STRATEGY = 'all'

RESULTS_PKL = CACHE_DIR / (
    f"other_organisms_deep_dive_snapshot{SNAPSHOT_RELEASE}_to{TARGET_RELEASES[0]}-{TARGET_RELEASES[-1]}_"
    f"n{N_SAMPLE_IDS}_strategy{STRATEGY}.pickle"
)

print('RESULTS_PKL:', RESULTS_PKL)


In [ ]:
# -------------------- Compute (cache-first) --------------------

import idtrack

if RESULTS_PKL.exists():
    payload = read_pickle(RESULTS_PKL)
    print('Loaded:', RESULTS_PKL)
else:
    rng = np.random.default_rng(0)

    api = idtrack.API(local_repository=str(IDTRACK_LOCAL_REPO))
    api.configure_logger()

    def reservoir_sample(nodes, prefix: str, k: int) -> list[str]:
        sample: list[str] = []
        n_seen = 0
        for node in nodes:
            if not isinstance(node, str) or not node.startswith(prefix):
                continue
            n_seen += 1
            if len(sample) < k:
                sample.append(node)
                continue
            j = int(rng.integers(0, n_seen))
            if j < k:
                sample[j] = node
        return sample

    rows = []
    for org in ORGANISMS:
        organism_alias = org['alias']
        prefix = org['prefix']

        organism, latest = api.resolve_organism(organism_alias)
        snapshot = min(int(SNAPSHOT_RELEASE), int(latest))
        api.build_graph(organism_name=organism, snapshot_release=snapshot, calculate_caches=True)

        ids = reservoir_sample(api.track.graph.nodes, prefix=prefix, k=N_SAMPLE_IDS)
        if not ids:
            raise RuntimeError(f'Could not sample IDs for {organism_alias} with prefix {prefix!r}.')

        for to_release in TARGET_RELEASES:
            matchings = api.convert_identifier_multiple(ids.copy(), to_release=int(to_release), final_database=None, strategy=STRATEGY)
            bins = api.classify_multiple_conversion(matchings)

            total = len(bins['input_identifiers'])
            n0 = len(bins['matching_1_to_0'])
            n1 = len(bins['matching_1_to_1']) + len(bins['alternative_target_1_to_1'])
            nn = len(bins['matching_1_to_n']) + len(bins['alternative_target_1_to_n'])

            rows.append(
                {
                    'organism': organism_alias,
                    'to_release': int(to_release),
                    'n': int(total),
                    '1_to_0': int(n0),
                    '1_to_1': int(n1),
                    '1_to_n': int(nn),
                }
            )

    payload = {'rows': rows}
    write_pickle(payload, RESULTS_PKL)
    print('Saved:', RESULTS_PKL)

df = pd.DataFrame(payload['rows'])
df.head()


In [ ]:
# -------------------- Plot: multi-panel manuscript-ready figure --------------------

df = pd.DataFrame(payload['rows'])

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.2), constrained_layout=True)
ax0, ax1 = axes

# Panel A: outcome profile at the median target release
mid_release = int(TARGET_RELEASES[len(TARGET_RELEASES) // 2])
sub = df[df['to_release'] == mid_release].set_index('organism')[['1_to_0', '1_to_1', '1_to_n']]
sub = sub.div(sub.sum(axis=1), axis=0)
sub.plot(
    kind='bar',
    stacked=True,
    ax=ax0,
    color=[MANUSCRIPT_COLORS['1→0'], MANUSCRIPT_COLORS['1→1'], MANUSCRIPT_COLORS['1→n']],
)
ax0.set_ylim(0, 1)
ax0.set_ylabel('Fraction of queries')
ax0.set_title(f'Outcome profile (to_release={mid_release})')
ax0.legend(['1→0', '1→1', '1→n'], loc='upper right', frameon=True)

# Panel B: time-axis trend (1→1 fraction)
for org in [o['alias'] for o in ORGANISMS]:
    d = df[df['organism'] == org].sort_values('to_release')
    frac = d['1_to_1'] / d['n'].replace(0, np.nan)
    ax1.plot(d['to_release'], frac, '-o', label=org)

ax1.set_ylim(0, 1)
ax1.set_xlabel('Target Ensembl release')
ax1.set_ylabel('1→1 fraction')
ax1.set_title('Stability under snapshot-bounded interpretation')
ax1.legend(frameon=True)

out_fig = MANUSCRIPT_FIGURES / 'fig_other_organisms_deep_dive.pdf'
fig.savefig(out_fig, bbox_inches='tight')
print('Saved:', out_fig)
